# Escenario extendido: los 4 modelos con variables recuperadas

Requiere haber ejecutado `notebooks/etl/01_features_ext.ipynb` para generar `data/processed/airprice_features_ext.csv`.

**Qué cambia respecto a geo:** se añaden variables del original que el ETL baseline descartó (review scores, disponibilidad, portafolio del host, amenidades, etc.) y geo adicional (densidad, distancias a estaciones/landmarks, K-Means k=5/8/10).

**Qué se mantiene:** mismo split, misma semilla y mismas métricas (`src.utils.evaluar_modelo` + métricas ordinales).

**Imputación:** el dataset extendido tiene nulos en review scores y fechas; el preprocesador imputa numéricas con la mediana y agrega indicadores de faltante. Los árboles también podrían manejar nulos, pero la imputación mantiene el pipeline homogéneo con el modelo ordinal.

**Target encoding:** no se usa por defecto en este notebook. Se evalúa en `02_tuning_extendido.ipynb` para no mezclar el efecto de las variables nuevas con el del codificador.

In [ ]:
import sys
sys.path.append('../..')

import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from src.model_features import (
    PARAMS_EXT,
    construir_modelo_features,
    evaluar_modelo_ordinal,
    importancia_variables,
    metricas_a_fila,
)
from src.model_ordinal import codificar_y, decodificar_y
from src.preprocessing_features import FEATURES_EXT_PATH, preparar_dataset_features
from src.utils import (
    CLASES,
    REPORTS_DIR,
    SEED,
    cargar_dataset,
    fijar_semilla,
    guardar_modelo,
    split_estratificado,
)

sns.set_theme(style='whitegrid')
fijar_semilla(SEED)
print('Semilla:', SEED)
print('Dataset:', FEATURES_EXT_PATH)

In [ ]:
df = preparar_dataset_features(
    cargar_dataset(FEATURES_EXT_PATH),
    descartar_baseline=False,
)
print('Filas:', df.shape[0], 'Columnas:', df.shape[1])

X_train, X_val, X_test, y_train, y_val, y_test = split_estratificado(
    df,
    target='price_range',
    excluir=['price_clean', 'id'],
    seed=SEED,
)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

geo_ext = [
    c for c in X_train.columns
    if c.startswith('distancia_') or c.startswith('cluster_zona')
]
print('Features geo extendidas:', geo_ext)
print('¿price_clean en X_train?', 'price_clean' in X_train.columns)
print('¿price en X_train?', 'price' in X_train.columns)

In [ ]:
resultados = []
modelos_entrenados = {}

for nombre in ['random_forest', 'xgboost', 'lightgbm', 'ordinal_logistic']:
    print('=' * 70)
    print('Entrenando:', nombre)
    params = dict(PARAMS_EXT[nombre])
    modelo = construir_modelo_features(
        nombre,
        X_train,
        imputar=True,
        target_encoding=None,
        **params,
    )

    if nombre in ('xgboost', 'ordinal_logistic'):
        modelo.fit(X_train, codificar_y(y_train))
        pred_val = decodificar_y(modelo.predict(X_val))
        pred_test = decodificar_y(modelo.predict(X_test))
    else:
        modelo.fit(X_train, y_train)
        pred_val = modelo.predict(X_val)
        pred_test = modelo.predict(X_test)

    met_val = evaluar_modelo_ordinal(y_val, pred_val)
    met_test = evaluar_modelo_ordinal(y_test, pred_test)
    print('Validación:', {k: round(v, 4) for k, v in met_val.items() if k != 'matriz_confusion'})
    print('Test:', {k: round(v, 4) for k, v in met_test.items() if k != 'matriz_confusion'})

    modelos_entrenados[nombre] = modelo
    resultados.append(
        metricas_a_fila(
            f'{nombre}_ext',
            met_test,
            escenario='extendido',
            **params,
        )
    )

    pd.DataFrame([resultados[-1]]).to_csv(
        REPORTS_DIR / f'{nombre}_ext_metricas.csv', index=False
    )

    ConfusionMatrixDisplay(
        met_test['matriz_confusion'], display_labels=CLASES
    ).plot(cmap='Blues', values_format='d')
    plt.title(f'{nombre} + extendido - test')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(
        REPORTS_DIR / f'{nombre}_ext_matriz_confusion.png',
        dpi=150,
        bbox_inches='tight',
    )
    plt.close()

    try:
        importancia = importancia_variables(modelo)
        importancia.to_csv(
            REPORTS_DIR / f'{nombre}_ext_importancia.csv', index=False
        )
    except Exception as error:
        print('No se pudo calcular importancia:', error)

    ruta = guardar_modelo(modelo, f'{nombre}_ext')
    print('Modelo guardado en:', ruta)

print('=' * 70)
print('Modelos entrenados:', list(modelos_entrenados))

In [ ]:
comparacion = pd.DataFrame(resultados).sort_values('f1_macro', ascending=False)
comparacion.to_csv(REPORTS_DIR / 'ext_comparacion_modelos.csv', index=False)
display(comparacion)

comparacion.set_index('modelo')[['accuracy', 'f1_macro', 'accuracy_adjacent']].plot(
    kind='bar', figsize=(10, 5)
)
plt.title('Escenario extendido: comparación de modelos')
plt.ylabel('Métrica')
plt.ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'ext_comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mejor = comparacion.iloc[0]
resumen = {
    'mejor_modelo': mejor['modelo'],
    'accuracy': float(mejor['accuracy']),
    'f1_macro': float(mejor['f1_macro']),
    'accuracy_adjacent': float(mejor['accuracy_adjacent']),
    'qwk': float(mejor['qwk']),
}
print(json.dumps(resumen, indent=2, ensure_ascii=False))

with open(REPORTS_DIR / 'ext_mejor_modelo.json', 'w', encoding='utf-8') as archivo:
    json.dump(resumen, archivo, indent=2, ensure_ascii=False)

## Lectura de resultados

- Comparar `accuracy` y `f1_macro` contra `reports/comparacion_base.csv` y `reports/comparacion_features.csv`.
- El escenario extendido no es estrictamente comparable con base/geo: añade variables y cambia el conjunto de features. Es el escenario de **mejor esfuerzo** para acercarse al 90 %.
- Si la accuracy exacta no llega a 0.90, revisar `accuracy_adjacent`: en un target ordinal de 4 clases, un acierto exacto o en la clase vecina suele ser la métrica que realmente importa al negocio.
- La importancia de variables y el tuning posterior indican qué variables nuevas aportan.